In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN

def load_house_data(csv_path):
    df = pd.read_csv(csv_path)
    if not {'latitude', 'longitude', 'tile_name'}.issubset(df.columns):
        raise ValueError("CSV must contain 'latitude', 'longitude', and 'tile_name'.")
    return df

def detect_poultry_farms(df, eps_house=50, min_samples=2, eps_merge=100):
    coords = df[['longitude', 'latitude']].values
    dbscan = DBSCAN(eps=eps_house, min_samples=min_samples).fit(coords)
    df['initial_cluster'] = dbscan.labels_

    clustered = df[df['initial_cluster'] != -1].copy()
    centroids = clustered.groupby('initial_cluster')[['longitude', 'latitude']].mean().reset_index()
    centroid_coords = centroids[['longitude', 'latitude']].values
    merge_cluster = DBSCAN(eps=eps_merge, min_samples=1).fit(centroid_coords)
    centroids['merged_farm_id'] = merge_cluster.labels_
    cluster_map = dict(zip(centroids['initial_cluster'], centroids['merged_farm_id']))
    clustered['merged_farm_id'] = clustered['initial_cluster'].map(cluster_map)
    df['merged_farm_id'] = np.where(df['initial_cluster'] == -1, -1, df['initial_cluster'].map(cluster_map))

    # Helper to format house coordinates
    def houses_string(lons, lats):
        return ','.join(f"({x},{y})" for x, y in zip(lons, lats))

    # Group by merged_farm_id and aggregate
    farm_groups = clustered.groupby('merged_farm_id')
    farms = farm_groups.agg(
        longitude=('longitude', 'mean'),
        latitude=('latitude', 'mean'),
        tile_name=('tile_name', lambda x: ','.join(sorted(set(x)))),
        num_houses=('longitude', 'size'),
        house_lons=('longitude', list),
        house_lats=('latitude', list)
    ).reset_index()

    # Compose houses column from house_lons and house_lats
    farms['houses'] = [houses_string(lons, lats) for lons, lats in zip(farms['house_lons'], farms['house_lats'])]
    farms = farms.drop(columns=['house_lons', 'house_lats'])

    return farms

def save_farms_to_csv(farms_df, output_path):
    farms_df.to_csv(output_path, index=False)
    print(f"✅ Saved farm centroids to: {output_path}")

# ===================== 🐔 PIPELINE EXECUTION =====================

input_csv = "CSV_Yolo_test/final_rect.csv"
output_csv = "CSV_Yolo_test/final_rect_highest_conf_farm.csv"

df = load_house_data(input_csv)
farm_centroids = detect_poultry_farms(df, eps_house=50, min_samples=2, eps_merge=100)
save_farms_to_csv(farm_centroids, output_csv)


✅ Saved farm centroids to: CSV_Yolo_test/final_rect_highest_conf_farm.csv


In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from tqdm import tqdm
import torch
from segment_anything import sam_model_registry, SamPredictor

# SAM Setup
CHECKPOINT_PATH = "weights/sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)
sam_predictor = SamPredictor(sam)

def lonlat_to_pixel(lon, lat, transform):
    # rowcol expects (x, y) = (lon, lat) if raster is WGS84
    row, col = rowcol(transform, lon, lat)
    return int(col), int(row)  # (x, y) in pixel space

def run_sam_on_lonlat_center_csv(
    csv_path, geotiff_dir, output_dir,
    class_label_col="class", conf_col="confidence",
    tile_col="tile_name",
    lon_col="longitude", lat_col="latitude",
    rect_halfsize=500
):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    tiff_files = {os.path.splitext(os.path.basename(f))[0]: f for f in os.listdir(geotiff_dir) if f.lower().endswith(('.tif', '.tiff'))}

    for tile_base, tiff_file in tqdm(tiff_files.items()):
        objects = df[df[tile_col] == (tile_base + ".tif")].to_dict(orient='records')
        if not objects:
            continue

        geotiff_path = os.path.join(geotiff_dir, tiff_file)
        with rasterio.open(geotiff_path) as src:
            img = src.read([1, 2, 3])
            img = np.transpose(img, (1, 2, 0))
            transform = src.transform

        if img.dtype != np.uint8:
            img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
        overlay = img.copy()

        for obj in objects:
            try:
                lon = float(obj[lon_col])
                lat = float(obj[lat_col])
                cx, cy = lonlat_to_pixel(lon, lat, transform)

                x1, x2 = cx - rect_halfsize, cx + rect_halfsize
                y1, y2 = cy - rect_halfsize, cy + rect_halfsize

                x1, x2 = np.clip([x1, x2], 0, overlay.shape[1] - 1)
                y1, y2 = np.clip([y1, y2], 0, overlay.shape[0] - 1)
                x1, x2 = int(x1), int(x2)
                y1, y2 = int(y1), int(y2)

                if x2 <= x1 or y2 <= y1:
                    continue

                roi_rgb = overlay[y1:y2, x1:x2]
                if roi_rgb.size == 0 or roi_rgb.shape[0] < 2 or roi_rgb.shape[1] < 2:
                    continue

                sam_predictor.set_image(roi_rgb)
                box = np.array([0, 0, x2-x1, y2-y1])
                masks, scores, logits = sam_predictor.predict(box=box, multimask_output=True)
                if masks is None or len(masks) == 0:
                    continue
                mask = masks[np.argmax(scores)]

                mask_full = np.zeros(overlay.shape[:2], dtype=np.uint8)
                mask_resized = cv2.resize(mask.astype(np.uint8), (x2-x1, y2-y1), interpolation=cv2.INTER_NEAREST)
                mask_full[y1:y2, x1:x2] = mask_resized

                mask_img = np.zeros_like(overlay)
                mask_img[mask_full > 0] = [0, 0, 255]
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.4, 0)

                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)
                label = f"Cls:{obj.get(class_label_col, '')} ({float(obj.get(conf_col, 0.0)):.2f})"
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 1.0
                thickness = 2
                text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
                text_origin = (x1, max(y1 - 5, text_size[1]))
                cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness+2)
                cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)
            except Exception:
                continue

        overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        out_path = os.path.join(output_dir, tile_base + "_sam_annotated.jpg")
        cv2.imwrite(out_path, overlay_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])

# ---------- Usage ----------
run_sam_on_lonlat_center_csv(
    csv_path="CSV_Yolo_test/final_rect_highest_conf_farm.csv",          # <-- update this path!
    geotiff_dir="test_sample",
    output_dir="annotated_geotiff_jpg"
)


100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [04:25<00:00, 26.58s/it]


In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import rowcol
from tqdm import tqdm
import torch
from segment_anything import sam_model_registry, SamPredictor

# --- SAM SETUP ---
CHECKPOINT_PATH = "weights/sam_vit_h_4b8939.pth"
MODEL_TYPE = "vit_h"
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)
sam_predictor = SamPredictor(sam)

# --- Convert lon/lat to pixel (x, y) ---
def lonlat_to_pixel(lon, lat, transform):
    row, col = rowcol(transform, lon, lat)
    return int(col), int(row)  # (x, y) in pixel space

def run_sam_on_lonlat_center_csv(
    csv_path, geotiff_dir, output_dir,
    class_label_col="class", conf_col="confidence",
    tile_col="tile_name",
    lon_col="longitude", lat_col="latitude",
    rect_halfsize=500  # pixels
):
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    tiff_files = {os.path.splitext(os.path.basename(f))[0]: f for f in os.listdir(geotiff_dir) if f.lower().endswith(('.tif', '.tiff'))}

    for tile_base, tiff_file in tqdm(tiff_files.items()):
        objects = df[df[tile_col] == (tile_base + ".tif")].to_dict(orient='records')
        if not objects:
            continue

        geotiff_path = os.path.join(geotiff_dir, tiff_file)
        with rasterio.open(geotiff_path) as src:
            img = src.read([1, 2, 3])  # Assumes RGB or first 3 bands are RGB
            img = np.transpose(img, (1, 2, 0))
            transform = src.transform

        # Normalize if not uint8
        if img.dtype != np.uint8:
            img = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
        overlay = img.copy()

        for obj in objects:
            try:
                lon = float(obj[lon_col])
                lat = float(obj[lat_col])
                cx, cy = lonlat_to_pixel(lon, lat, transform)

                x1, x2 = cx - rect_halfsize, cx + rect_halfsize
                y1, y2 = cy - rect_halfsize, cy + rect_halfsize

                x1, x2 = np.clip([x1, x2], 0, overlay.shape[1] - 1)
                y1, y2 = np.clip([y1, y2], 0, overlay.shape[0] - 1)
                x1, x2 = int(x1), int(x2)
                y1, y2 = int(y1), int(y2)

                if x2 <= x1 or y2 <= y1:
                    continue

                roi_rgb = overlay[y1:y2, x1:x2]
                if roi_rgb.size == 0 or roi_rgb.shape[0] < 2 or roi_rgb.shape[1] < 2:
                    continue

                sam_predictor.set_image(roi_rgb)
                # Center point in crop coords
                roi_cx, roi_cy = (x2-x1)//2, (y2-y1)//2
                # Both box and point prompt for best accuracy!
                masks, scores, logits = sam_predictor.predict(
                    box=np.array([0, 0, x2-x1, y2-y1]),
                    point_coords=np.array([[roi_cx, roi_cy]]),
                    point_labels=np.array([1]),
                    multimask_output=True
                )
                if masks is None or len(masks) == 0:
                    continue
                mask = masks[np.argmax(scores)]

                mask_full = np.zeros(overlay.shape[:2], dtype=np.uint8)
                mask_resized = cv2.resize(mask.astype(np.uint8), (x2-x1, y2-y1), interpolation=cv2.INTER_NEAREST)
                mask_full[y1:y2, x1:x2] = mask_resized

                mask_img = np.zeros_like(overlay)
                mask_img[mask_full > 0] = [0, 0, 255]  # Red mask
                overlay = cv2.addWeighted(overlay, 1.0, mask_img, 0.4, 0)

                # Draw bbox and label
                cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)
                label = f"Cls:{obj.get(class_label_col, '')} ({float(obj.get(conf_col, 0.0)):.2f})"
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 1.0
                thickness = 2
                text_size = cv2.getTextSize(label, font, font_scale, thickness)[0]
                text_origin = (x1, max(y1 - 5, text_size[1]))
                cv2.putText(overlay, label, text_origin, font, font_scale, (0, 0, 0), thickness+2)
                cv2.putText(overlay, label, text_origin, font, font_scale, (255, 255, 255), thickness)
            except Exception as e:
                print(f"Error: {e}")
                continue

        overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
        out_path = os.path.join(output_dir, tile_base + "_sam_annotated.png")
        cv2.imwrite(out_path, overlay_bgr)
        print(f"Saved: {out_path}")

# ---------- USAGE ----------
run_sam_on_lonlat_center_csv(
    csv_path="CSV_Yolo_test/final_rect_highest_conf_farm.csv",    # <-- your CSV
    geotiff_dir="test_sample",                                    # <-- your geotiff dir
    output_dir="annotated_geotiff_png",                           # <-- output dir
    rect_halfsize=500                                             # <-- crop halfsize in pixels; adjust as needed
)


 10%|████████▍                                                                           | 1/10 [00:42<06:22, 42.47s/it]

Saved: annotated_geotiff_png/m_3108502_sw_16_030_20211103_sam_annotated.png


 20%|████████████████▊                                                                   | 2/10 [01:13<04:47, 35.88s/it]

Saved: annotated_geotiff_png/m_3108501_se_16_030_20211103_sam_annotated.png


 30%|█████████████████████████▏                                                          | 3/10 [01:54<04:28, 38.31s/it]

...

Saved: annotated_geotiff_png/m_3108502_ne_16_030_20211103_sam_annotated.png


 90%|███████████████████████████████████████████████████████████████████████████▌        | 9/10 [05:57<00:41, 41.90s/it]

Saved: annotated_geotiff_png/m_3108501_ne_16_030_20211103_sam_annotated.png


100%|███████████████████████████████████████████████████████████████████████████████████| 10/10 [06:26<00:00, 38.68s/it]

Saved: annotated_geotiff_png/m_3108501_nw_16_030_20211103_sam_annotated.png
